# Global Moran's I: an interactive lab

This notebook turns the formula into something you can inspect. We will build a tiny neighbourhood graph, calculate the statistic, see how weights change the result, and compare the observed value with every possible reassignment of four observations.

**You need:** Python, NumPy and Matplotlib. This is a teaching example, not a substitute for a full spatial inference workflow.

The statistic is

$$
I = \frac{n}{S_0}\frac{\sum_i\sum_j w_{ij}z_i z_j}{\sum_i z_i^2}
$$

where $z_i=x_i-\bar{x}$ and $S_0=\sum_i\sum_j w_{ij}$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import permutations


## 1. Make a small neighbourhood graph

Four areas form a chain. Each area is connected to the area immediately beside it. The matrix is symmetric, has a zero diagonal and uses binary weights.

In [ ]:
W_binary = np.array([
    [0, 1, 0, 0],
    [1, 0, 1, 0],
    [0, 1, 0, 1],
    [0, 0, 1, 0],
], dtype=float)

values = np.array([1, 2, 3, 4], dtype=float)

def morans_i(values, weights):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    n = len(values)
    z = values - values.mean()
    s0 = weights.sum()
    return (n / s0) * ((z @ weights @ z) / (z @ z))

print('Values:', values)
print('Binary weights Moran I:', morans_i(values, W_binary))


In [ ]:
positions = np.arange(len(values))
fig, ax = plt.subplots(figsize=(9, 2.6))

for i in range(len(values) - 1):
    ax.plot([positions[i], positions[i + 1]], [0, 0], linewidth=3)

ax.scatter(positions, np.zeros_like(positions), s=950, zorder=3)
for i, value in enumerate(values):
    ax.text(positions[i], 0, f'Area {i + 1}\nvalue = {value:g}', ha='center', va='center', color='white', weight='bold')

ax.set_xlim(-0.5, 3.5)
ax.set_ylim(-0.65, 0.65)
ax.set_yticks([])
ax.set_xticks([])
ax.set_title('Four areas and their neighbour relationships')
ax.axis('off')
plt.show()


## 2. Compare binary and row-standardised weights

Row standardisation divides each nonzero row by its sum. Every area's neighbour weights then sum to one. An area with no neighbours would remain a zero row.

In [ ]:
row_sums = W_binary.sum(axis=1, keepdims=True)
W_row = np.divide(W_binary, row_sums, out=np.zeros_like(W_binary), where=row_sums != 0)

print('Binary weights:')
print(W_binary)
print('Row-standardised weights:')
print(W_row)
print('Moran I with binary weights:', morans_i(values, W_binary))
print('Moran I with row-standardised weights:', morans_i(values, W_row))


Notice that the result changes even though the map and values do not. The weights matrix is part of the model, not a neutral technical setting.

## 3. See the spatial lag

The spatial lag is the weighted value of each area's neighbours. With row-standardised weights it is a neighbourhood average. The slope of the fitted line through the centred values and their spatial lag equals global Moran's I for this row-standardised matrix.

In [ ]:
z = values - values.mean()
spatial_lag = W_row @ z
observed_i = morans_i(values, W_row)
slope = (z @ spatial_lag) / (z @ z)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(z, spatial_lag, s=90)
x_line = np.linspace(z.min() - 0.2, z.max() + 0.2, 100)
ax.plot(x_line, slope * x_line, linewidth=2)
for i, (x, y) in enumerate(zip(z, spatial_lag), start=1):
    ax.annotate(f'Area {i}', (x, y), xytext=(5, 5), textcoords='offset points')
ax.axhline(0, linewidth=0.8)
ax.axvline(0, linewidth=0.8)
ax.set_xlabel('Area value minus overall mean')
ax.set_ylabel('Spatial lag of centred values')
ax.set_title(f'Moran scatterplot; slope = {slope:.3f}, I = {observed_i:.3f}')
plt.show()


## 4. Compare with a reference distribution

Because there are only four areas, we can enumerate all $4! = 24$ ways to assign these same values to the fixed locations. This gives an exact permutation reference for this tiny example. For larger datasets, all permutations are impossible to enumerate; Monte Carlo permutations are used instead.

In [ ]:
all_permutations = list(permutations(values))
simulated_i = np.array([morans_i(permuted, W_row) for permuted in all_permutations])
exact_right_tail_p = np.mean(simulated_i >= observed_i)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(simulated_i, bins=10)
ax.axvline(observed_i, linewidth=2, label=f'Observed I = {observed_i:.3f}')
ax.set_xlabel("Moran's I under reassignment")
ax.set_ylabel('Number of arrangements')
ax.set_title(f'All 24 assignments; exact right-tail p = {exact_right_tail_p:.3f}')
ax.legend()
plt.show()

print('Number of arrangements:', len(simulated_i))
print('Exact right-tail p-value:', exact_right_tail_p)


For a Monte Carlo test with $M$ random permutations, a common one-sided pseudo-$p$-value uses the correction

$$
p = \frac{C+1}{M+1}
$$

where $C$ is the number of simulated statistics at least as large as the observed statistic. Choose the test direction before looking at the result.

## 5. Change the pattern

Replace the values with `[1, 4, 1, 4]` and rerun the relevant cells. The neighbourhood graph stays the same, but high and low values alternate. What happens to Moran's I and the scatterplot?

In [ ]:
alternating_values = np.array([1, 4, 1, 4], dtype=float)
print('Alternating values:', alternating_values)
print('Moran I with binary weights:', morans_i(alternating_values, W_binary))
print('Moran I with row-standardised weights:', morans_i(alternating_values, W_row))


## What to take away

- The weights matrix defines the neighbourhood relationship being tested.
- Global Moran's I summarises the whole pattern; it does not locate local clusters.
- The statistic and its significance are different things. Use a reference distribution and state the null hypothesis.
- A change in weights can change the result without changing the observed values.

Continue with the [canonical Global Moran's I lesson](../algorithms/statistics/global-morans-i.md) and its [tested Python implementation](../implementations/python/global-morans-i/README.md).